In [ ]:
from toy_models.create import ToyMultipartite
from models import LP
toy = ToyMultipartite()
solution = LP(toy,model_type='zerosum_mf_static_nodes').solve()
drawer = solution.draw()
analysis = solution.analyze()
ml = analysis.ml()

100%|██████████| 50/50 [00:00<00:00, 217.93it/s]


In [ ]:
import pytest

from toy_models.create import ToyMultipartite
from models import LP, Solver
from analysis import Analysis
from analysis import GraphMetrics


@pytest.fixture
def toy_graph():
    return ToyMultipartite(layer_list=[1, 2, 2, 1], seed=123)

@pytest.fixture
def lp(toy_graph):
    return LP(toy_graph.G, model_type="max_flow")

@pytest.fixture
def solver(lp):
    return Solver(lp)

@pytest.fixture
def analysis(solver):
    return Analysis(solver)

@pytest.fixture
def graph_metrics(analysis):
    return GraphMetrics(analysis)

In [ ]:
def test_graph_info(graph_metrics):
    info = graph_metrics.graph_info()

    assert info["nodes"] == 6
    assert info["directed"] is True

def test_degree_metrics(graph_metrics):
    df = graph_metrics.degree_metrics()

    assert len(df) == 6
    assert "node" in df.columns
    assert "degree" in df.columns

def test_graph_info(graph_metrics):
    info = graph_metrics.graph_info()

    assert info["nodes"] == graph_metrics.g.number_of_nodes()
    assert info["edges"] == graph_metrics.g.number_of_edges()
    assert info["directed"] == graph_metrics.g.is_directed()

def test_degree_metrics(graph_metrics):
    df = graph_metrics.degree_metrics()
    assert len(df) == graph_metrics.g.number_of_nodes()
    assert df["node"].is_unique
    for node in graph_metrics.g.nodes:
        row = df.loc[df["node"] == node].iloc[0]
        assert row["degree"] == graph_metrics.g.degree(node)


def test_centrality_metrics(graph_metrics):
    df = graph_metrics.centrality_metrics()
    expected_columns = {"node", "degree_centrality", "betweenness_centrality", "closeness_centrality", "eigenvector_centrality",}
    assert expected_columns.issubset(df.columns)
    assert len(df) == graph_metrics.g.number_of_nodes()
    assert df["node"].is_unique
    assert df["eigenvector_centrality"].notna().all()

def test_spectral_metrics(graph_metrics):
    metrics = graph_metrics.spectral_metrics()

    assert metrics["adjacency_spectral_radius"] >= 0
    assert metrics["laplacian_algebraic_connectivity"] >= 0

def test_centrality_no_nan(graph_metrics):
    df = graph_metrics.centrality_metrics()

    assert df["eigenvector_centrality"].notna().all()

